# Batch Inference via the Batch Gateway

The **llm-d Batch Gateway** provides an OpenAI-compatible batch processing
API with `/v1/files` and `/v1/batches` endpoints. This lets you submit
large sets of inference requests as a single batch job, then retrieve
results when processing is complete.

Unlike the Async Processor (which uses Redis queues directly), the Batch
Gateway provides a higher-level API that is compatible with existing
OpenAI batch processing code. It handles:

- Uploading input files with multiple requests
- Creating and managing batch jobs
- Tracking job progress and status
- Downloading results as output files
- Flow control to protect interactive workloads

This notebook covers deploying the Batch Gateway, submitting batch jobs
through the API, and monitoring their execution on shared infrastructure.

In [ ]:
# Environment setup
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["MODEL_NAME"] = "Qwen/Qwen3-32B"

print("Namespace:", os.environ["NAMESPACE"])
print("Model:", os.environ["MODEL_NAME"])

## Batch Gateway Architecture

The Batch Gateway sits alongside the interactive Envoy Gateway and
shares the same vLLM backend through the EPP router.

```
Interactive Clients           Batch Clients
       |                           |
       v                           v
+-------------+           +-----------------+
| Envoy       |           | Batch Gateway   |
| Gateway     |           | /v1/files       |
| /v1/chat/   |           | /v1/batches     |
| completions |           +---------+-------+
+------+------+                     |
       |                            |
       v                            v
+----------------------------------+
| EPP Router (with flow control)   |
| Interactive: priority band 1     |
| Batch:       priority band 3     |
+------+---------------------------+
       |
       v
+-------------+
| vLLM        |
| Replicas    |
+-------------+
```

Flow control priority bands ensure interactive traffic always takes
precedence. Batch requests are placed in a lower priority band and
are only processed when there is spare capacity.

In [ ]:
# Step 1: Deploy the llm-d Batch Gateway

batch_gw_manifest = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: batch-gateway
  namespace: llm-d
spec:
  replicas: 1
  selector:
    matchLabels:
      app: batch-gateway
  template:
    metadata:
      labels:
        app: batch-gateway
    spec:
      containers:
        - name: batch-gateway
          image: ghcr.io/llm-d/batch-gateway:latest
          ports:
            - containerPort: 8080
              name: http
          env:
            - name: INFERENCE_URL
              value: http://envoy-gateway.llm-d.svc:8080
            - name: STORAGE_BACKEND
              value: pvc
            - name: STORAGE_PATH
              value: /data/batch-files
            - name: CONCURRENCY
              value: "16"
            - name: PRIORITY_BAND
              value: "3"
          volumeMounts:
            - name: batch-storage
              mountPath: /data/batch-files
          resources:
            requests:
              memory: 1Gi
              cpu: "1"
      volumes:
        - name: batch-storage
          persistentVolumeClaim:
            claimName: batch-storage-pvc
---
apiVersion: v1
kind: Service
metadata:
  name: batch-gateway
  namespace: llm-d
spec:
  selector:
    app: batch-gateway
  ports:
    - port: 8080
      targetPort: 8080
  type: LoadBalancer
---
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: batch-storage-pvc
  namespace: llm-d
spec:
  accessModes:
    - ReadWriteOnce
  resources:
    requests:
      storage: 50Gi
"""

with open("/tmp/batch-gateway.yaml", "w") as f:
    f.write(batch_gw_manifest)

!kubectl apply -f /tmp/batch-gateway.yaml

print("Batch Gateway deployed.")
!kubectl wait --for=condition=ready pod -l app=batch-gateway -n $NAMESPACE --timeout=120s
print("Batch Gateway is ready.")
print()
print("Endpoints:")
print("  POST   /v1/files          Upload input files")
print("  POST   /v1/batches        Create batch jobs")
print("  GET    /v1/batches/{id}    Check batch status")
print("  GET    /v1/files/{id}      Download result files")

In [ ]:
# Step 2: Create an input file with multiple requests
# The input file uses JSONL format (one request per line),
# following the OpenAI batch file format

import subprocess
import json

BATCH_GW_IP = subprocess.check_output(
    "kubectl get svc batch-gateway -n llm-d "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

print(f"Batch Gateway: {BATCH_GW_IP}:8080")
print()

# Create the input JSONL file
requests = [
    {
        "custom_id": f"request-{i+1}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "Qwen/Qwen3-32B",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 200,
        },
    }
    for i, prompt in enumerate([
        "What is the theory of relativity?",
        "Explain photosynthesis in simple terms.",
        "What causes earthquakes?",
        "How does the internet work?",
        "What is DNA and why is it important?",
        "Explain the water cycle.",
        "What is inflation in economics?",
        "How do vaccines work?",
        "What is machine learning?",
        "Explain quantum computing in simple terms.",
        "What is climate change?",
        "How does a combustion engine work?",
        "What is the Big Bang theory?",
        "Explain how airplanes fly.",
        "What is blockchain technology?",
        "How does the human immune system work?",
        "What is dark matter?",
        "Explain the greenhouse effect.",
        "What is nuclear fusion?",
        "How do solar panels generate electricity?",
    ])
]

# Write the JSONL file
input_file = "\n".join(json.dumps(r) for r in requests)
with open("/tmp/batch-input.jsonl", "w") as f:
    f.write(input_file)

print(f"Created input file with {len(requests)} requests.")
print(f"File size: {len(input_file)} bytes")
print()
print("Sample request:")
print(json.dumps(requests[0], indent=2))

In [ ]:
# Step 3: Upload the input file using /v1/files

result = subprocess.run(
    ["curl", "-s", f"http://{BATCH_GW_IP}:8080/v1/files",
     "-F", "purpose=batch",
     "-F", "file=@/tmp/batch-input.jsonl"],
    capture_output=True, text=True
)

file_response = json.loads(result.stdout)
file_id = file_response.get("id", "unknown")

print("File uploaded successfully.")
print(json.dumps(file_response, indent=2))
print()
print(f"File ID: {file_id}")
print("Use this ID to create a batch job.")

In [ ]:
# Step 4: Create a batch job using /v1/batches

batch_request = {
    "input_file_id": file_id,
    "endpoint": "/v1/chat/completions",
    "completion_window": "24h",
}

result = subprocess.run(
    ["curl", "-s", f"http://{BATCH_GW_IP}:8080/v1/batches",
     "-H", "Content-Type: application/json",
     "-d", json.dumps(batch_request)],
    capture_output=True, text=True
)

batch_response = json.loads(result.stdout)
batch_id = batch_response.get("id", "unknown")

print("Batch job created.")
print(json.dumps(batch_response, indent=2))
print()
print(f"Batch ID: {batch_id}")
print(f"Status: {batch_response.get('status', 'unknown')}")
print(f"Total requests: {batch_response.get('request_counts', {}).get('total', 'N/A')}")

In [ ]:
# Step 5: Monitor batch job status
import time

print(f"Monitoring batch job {batch_id}...")
print()

for i in range(20):
    result = subprocess.run(
        ["curl", "-s", f"http://{BATCH_GW_IP}:8080/v1/batches/{batch_id}"],
        capture_output=True, text=True
    )

    status = json.loads(result.stdout)
    counts = status.get("request_counts", {})
    completed = counts.get("completed", 0)
    failed = counts.get("failed", 0)
    total = counts.get("total", 0)
    batch_status = status.get("status", "unknown")

    progress = (completed + failed) / total * 100 if total > 0 else 0

    print(f"  [{(i+1)*15:>3}s] Status: {batch_status:<12} "
          f"Progress: {completed}/{total} ({progress:.0f}%) "
          f"Failed: {failed}")

    if batch_status in ("completed", "failed", "expired", "cancelled"):
        print(f"\n  Batch job {batch_status}.")
        break

    time.sleep(15)

# Show final status
print()
print("Final batch status:")
print(json.dumps(status, indent=2))

In [ ]:
# Step 6: Download results when complete

output_file_id = status.get("output_file_id", None)

if output_file_id:
    print(f"Downloading results (file ID: {output_file_id})...")
    print()

    result = subprocess.run(
        ["curl", "-s",
         f"http://{BATCH_GW_IP}:8080/v1/files/{output_file_id}/content"],
        capture_output=True, text=True
    )

    # Parse JSONL output
    lines = result.stdout.strip().split("\n")
    print(f"Received {len(lines)} results.")
    print()

    # Show first 3 results
    for i, line in enumerate(lines[:3]):
        output = json.loads(line)
        custom_id = output.get("custom_id", "unknown")
        response = output.get("response", {})
        body = response.get("body", {})

        print(f"--- Result {custom_id} ---")
        if "choices" in body:
            content = body["choices"][0]["message"]["content"]
            print(f"  {content[:200]}...")
        print(f"  Status: {response.get('status_code', 'N/A')}")
        print()

    if len(lines) > 3:
        print(f"  ... and {len(lines) - 3} more results.")
else:
    print("No output file available yet. The batch may still be processing.")
    print(f"Check status: curl http://{BATCH_GW_IP}:8080/v1/batches/{batch_id}")

## Flow Control and Priority Bands

Batch jobs coexist with interactive workloads on the same GPU
infrastructure. The EPP uses **priority bands** to ensure interactive
traffic is never impacted:

| Priority Band | Traffic Type | Behavior |
|--------------|--------------|----------|
| 1 (highest)  | Interactive  | Always processed first |
| 2            | Near-real-time | Processed when band 1 has spare capacity |
| 3 (lowest)   | Batch        | Only processed when bands 1 and 2 are idle |

The Batch Gateway sends all requests in priority band 3 by default.
When interactive load spikes, batch processing automatically slows
down. When interactive load drops, batch processing speeds up to
use the available capacity.

This means you can submit large batch jobs without worrying about
impacting interactive users. The GPUs are fully utilized either way.

In [ ]:
# Demonstrate batch and interactive coexistence
# Submit a new batch while running interactive queries

import threading

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

# Create and submit a new batch job
print("Creating a new batch job (30 requests)...")
batch_requests_2 = []
for i in range(30):
    batch_requests_2.append(json.dumps({
        "custom_id": f"coexist-{i+1}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "Qwen/Qwen3-32B",
            "messages": [{"role": "user",
                          "content": f"Write a short fact about the number {i+1}."}],
            "max_tokens": 100,
        },
    }))

with open("/tmp/batch-input-2.jsonl", "w") as f:
    f.write("\n".join(batch_requests_2))

# Upload and create batch
upload = subprocess.run(
    ["curl", "-s", f"http://{BATCH_GW_IP}:8080/v1/files",
     "-F", "purpose=batch",
     "-F", "file=@/tmp/batch-input-2.jsonl"],
    capture_output=True, text=True
)
file_id_2 = json.loads(upload.stdout).get("id")

create = subprocess.run(
    ["curl", "-s", f"http://{BATCH_GW_IP}:8080/v1/batches",
     "-H", "Content-Type: application/json",
     "-d", json.dumps({
         "input_file_id": file_id_2,
         "endpoint": "/v1/chat/completions",
         "completion_window": "24h",
     })],
    capture_output=True, text=True
)
batch_id_2 = json.loads(create.stdout).get("id")
print(f"Batch job created: {batch_id_2}")
print()

# Run interactive queries simultaneously
interactive_ttfts = []
stop = threading.Event()

def run_interactive():
    while not stop.is_set():
        start = time.time()
        subprocess.run(
            ["curl", "-s", "-m", "30",
             f"http://{GATEWAY_IP}:8080/v1/chat/completions",
             "-H", "Content-Type: application/json",
             "-d", json.dumps({
                 "model": "Qwen/Qwen3-32B",
                 "messages": [{"role": "user", "content": "Hello!"}],
                 "max_tokens": 10,
             })],
            capture_output=True
        )
        interactive_ttfts.append(time.time() - start)

# Run 5 interactive threads for 60 seconds
print("Running 5 interactive threads alongside batch processing...")
threads = [threading.Thread(target=run_interactive, daemon=True) for _ in range(5)]
for t in threads:
    t.start()

for i in range(6):
    time.sleep(10)
    batch_status = subprocess.run(
        ["curl", "-s", f"http://{BATCH_GW_IP}:8080/v1/batches/{batch_id_2}"],
        capture_output=True, text=True
    )
    bs = json.loads(batch_status.stdout)
    bc = bs.get("request_counts", {}).get("completed", 0)
    bt = bs.get("request_counts", {}).get("total", 0)

    recent = interactive_ttfts[-5:] if interactive_ttfts else [0]
    avg_ttft = sum(recent) / len(recent) * 1000

    print(f"  [{(i+1)*10:>2}s] Batch: {bc}/{bt}, "
          f"Interactive TTFT: {avg_ttft:.0f}ms, "
          f"Interactive total: {len(interactive_ttfts)} requests")

stop.set()
print()
print("Interactive TTFT remained low throughout batch processing,")
print("demonstrating that flow control priority bands are working correctly.")

## Summary

The Batch Gateway provides an OpenAI-compatible API for submitting
and managing large-scale batch inference jobs on shared llm-d
infrastructure.

### API Reference

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/v1/files` | POST | Upload a JSONL input file |
| `/v1/files/{id}/content` | GET | Download file content (input or output) |
| `/v1/batches` | POST | Create a new batch job |
| `/v1/batches/{id}` | GET | Get batch job status and progress |
| `/v1/batches/{id}/cancel` | POST | Cancel a running batch job |

### Workflow

1. Create a JSONL input file (one request per line)
2. Upload via `POST /v1/files`
3. Create batch via `POST /v1/batches` with the file ID
4. Poll `GET /v1/batches/{id}` until status is "completed"
5. Download results via `GET /v1/files/{output_file_id}/content`

### Key Features

- **OpenAI-compatible**: Drop-in replacement for OpenAI's batch API
- **Shared infrastructure**: Batch and interactive on the same GPUs
- **Flow control**: Priority bands prevent batch from impacting interactive
- **Persistent storage**: Input and output files stored on PVC

### Next Steps

- **Async Processing**: For lower-level Redis-based queue processing.
- **Flow Control**: Fine-tune priority bands and concurrency limits.
- **Autoscaling**: Scale GPU replicas based on combined interactive
  and batch demand.